## Import Necessary Libraries

In [1]:
import math
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F

## Scaled Dot Product Attention

In [ ]:
def scaled_dot_product_attention(
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    mask: torch.Tensor | None = None,
    dropout: nn.Dropout | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute Scaled Dot-Product Attention.

    Args:
        query:   (batch, heads, seq_len_q, d_k)
        key:     (batch, heads, seq_len_k, d_k)
        value:   (batch, heads, seq_len_k, d_v)
        mask:    Broadcastable mask. 0/False positions are *allowed*,
                 True positions are *blocked* (filled with -inf before softmax).
        dropout: Optional dropout applied to the attention weights.

    Returns:
        output:  (batch, heads, seq_len_q, d_v)
        attn_weights: (batch, heads, seq_len_q, seq_len_k)
    """
    d_k = query.size(-1)

    # Step 1: Compute raw attention scores  →  Q K^T
    # Shape: (batch, heads, seq_len_q, seq_len_k)
    scores = torch.matmul(query, key.transpose(-2, -1))

    # Step 2: Scale by √d_k to stabilise gradients
    scores = scores / math.sqrt(d_k)

    # Step 3: Apply mask (used for padding and causal/autoregressive decoding)
    # Masked positions get -inf so that softmax assigns them ~0 probability.
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))

    # Step 4: Softmax along the key dimension → attention weights sum to 1
    attn_weighst = F.softmax(scores, dim=-1)

    # Step 5: Optional dropout on attention weights (Section 5.4 in the paper)
    if dropout is not None:
        attn_weights = dropout(attn_weights)

    # Step 6: Weighted sum of values
    output = torch.matmul(attn_weights, value)

    return output, attn_weights

## Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention mechanism (Section 3.2.2).

    This module is used in three different ways in the Transformer:
      1. Encoder self-attention — Q=K=V come from encoder input
      2. Decoder self-attention — Q=K=V come from decoder input (masked)
      3. Encoder-decoder attention — Q from decoder, K=V from encoder output
    """

    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Dimension per head

        # Learned linear projections: W_Q, W_K, W_V, W_O
        # Each is (d_model, d_model) but conceptually splits into h × (d_model, d_k)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)  # Output projection

        self.dropout = nn.Dropout(dropout)
        self.attn_weights = None  # Stored for visualisation

    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """
        Args:
            query: (batch, seq_len_q, d_model)
            key:   (batch, seq_len_k, d_model)
            value: (batch, seq_len_k, d_model)
            mask:  (batch, 1, 1, seq_len_k) or (batch, 1, seq_len_q, seq_len_k)

        Returns:
            output: (batch, seq_len_q, d_model)
        """
        batch_size = query.size(0)

        # ── Step 1: Linear projections ──────────────────────────────────────
        # Project and reshape: (batch, seq_len, d_model) → (batch, heads, seq_len, d_k)
        # This splits d_model into num_heads separate d_k-dimensional subspaces.
        Q = self.W_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # ── Step 2: Apply scaled dot-product attention to all heads in parallel ─
        # Thanks to the batch dimension including heads, this is done in one
        # matmul — no loop over heads needed.
        x, self.attn_weights = scaled_dot_product_attention(
            Q, K, V, mask=mask, dropout=self.dropout
        )

        # ── Step 3: Concatenate heads and apply output projection ───────────
        # (batch, heads, seq_len, d_k) → (batch, seq_len, d_model)
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

        # Final linear: W^O in Equation (5)
        return self.W_o(x)

## Position-wise Feed-Forward Networks

In [ ]:
class PositionwiseFeedForward(nn.Module):
    """Position-wise Feed-Forward Network (Section 3.3)."""

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)     # W_1: expand
        self.linear2 = nn.Linear(d_ff, d_model)      # W_2: compress back
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, d_model)
        return self.linear2(self.dropout(F.relu(self.linear1(x))))